In [1]:
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import classification_report, multilabel_confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import copy

# OpenMP 충돌 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
Config = {
    "NUM_CLASSES": 5,         # 0 normal, 1 dos, 2 fuzzing, 3 replay, 4 spoofing
    "BATCH_SIZE": 64,
}

In [3]:
# ================================
# 1. Causal Convolution 레이어 정의
# ================================
class CausalConv1d(nn.Module):
    """
    미래의 데이터를 보지 않도록 왼쪽으로만 패딩을 넣는 컨볼루션 레이어입니다.
    """
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super(CausalConv1d, self).__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)

    def forward(self, x):
        x = self.conv(x)
        # 오른쪽 끝부분(미래 데이터 위치)을 잘라내어 인과성을 유지합니다.
        if self.padding != 0:
            x = x[:, :, :-self.padding]
        return x

In [4]:
# ================================
# 2. 업데이트된 SeqIDS 모델 (Causal 적용)
# ================================
class SeqIDS(nn.Module):
    def __init__(self, num_classes=5):
        super(SeqIDS, self).__init__()
        # 기존 Sequential 대신 CausalConv1d를 사용하도록 구성
        self.layer1 = CausalConv1d(9, 32, kernel_size=5)
        self.relu1 = nn.ReLU()
        self.layer2 = CausalConv1d(32, 64, kernel_size=3)
        self.relu2 = nn.ReLU()
        
        # 각 패킷(time step)별로 클래스를 분류하는 1x1 Conv
        self.classifier = nn.Conv1d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # x: (B, 9, 64)
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        
        logits = self.classifier(x)  # (B, 5, 64)
        return logits


In [5]:
class SeqWindowDataset(Dataset):
    def __init__(self, tensor_X, tensor_y):
        # 혹시 numpy가 들어오더라도 방어적으로 처리
        if isinstance(tensor_X, np.ndarray):
            tensor_X = torch.tensor(tensor_X, dtype=torch.float32)
        if isinstance(tensor_y, np.ndarray):
            tensor_y = torch.tensor(tensor_y, dtype=torch.long)

        self.X = tensor_X  # (N, 9, 64)
        self.y = tensor_y  # (N, 64)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        return x, y


In [6]:
# ================================
# 4. 학습 및 평가 루틴
# ================================
def evaluate_window_level(model, loader, device, target_names=None):
    model.eval()
    all_preds, all_targets = [] , []

    # 클래스 이름이 안 들어오면 기본값 사용
    if target_names is None:
        target_names = ['Normal', 'DoS', 'Fuzzy', 'Replay', 'Spoofing']

    # 클래스 인덱스 0~4에 대해 멀티라벨 바이너리화
    mlb = MultiLabelBinarizer(classes=[0, 1, 2, 3, 4])

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)

            # 예측 및 정답 추출
            preds_indices = logits.argmax(dim=1) # (B, 64)
            
            for b in range(y_batch.shape[0]):
                # 예측값 처리
                p_unique = torch.unique(preds_indices[b]).tolist()
                if len(p_unique) > 1 and 0 in p_unique:
                    p_unique.remove(0)
                all_preds.append(p_unique)

                # 정답값 처리
                t_unique = torch.unique(y_batch[b]).tolist()
                if len(t_unique) > 1 and 0 in t_unique:
                    t_unique.remove(0)
                all_targets.append(t_unique)

    # 이진 행렬 변환
    y_true = mlb.fit_transform(all_targets)
    y_pred = mlb.transform(all_preds)

    print("\n=== Window-Level Multi-Label Evaluation Results ===")
    print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))
    
    return multilabel_confusion_matrix(y_true, y_pred)



In [7]:
def train_and_validate(train_data_path, model_save_path, epochs=20, lr=1e-3, device=None):
    """
    DATA_PATH의 데이터를 로드하여 Train/Val로 나누고 학습 및 검증을 수행합니다.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥 실행 디바이스: {device}")

    # 1. 학습 데이터 로드
    print(f"📂 학습 데이터셋 로딩 중: {train_data_path}")
    data = np.load(train_data_path)

    X_np = data["X"]
    y_np = data["y"]

    print("X shape:", X_np.shape, "y shape:", y_np.shape)
    print("X dtype:", X_np.dtype, "y dtype:", y_np.dtype)

    # NaN / Inf 체크 (중요!!)
    print("X has NaN:", np.isnan(X_np).any(), "| X has Inf:", np.isinf(X_np).any())
    print("y has NaN:", np.isnan(y_np).any(), "| y has Inf:", np.isinf(y_np).any())

    # numpy -> torch 명시 변환 + dtype 통일
    X_tensor = torch.tensor(X_np, dtype=torch.float32)
    y_tensor = torch.tensor(y_np, dtype=torch.long)

    # 혹시라도 NaN/Inf가 있으면 안전하게 처리 (선택)
    X_tensor = torch.nan_to_num(X_tensor, nan=0.0, posinf=1e6, neginf=-1e6)

    full_dataset = SeqWindowDataset(X_tensor, y_tensor)

    
    # 2. 데이터셋 분할 (Train : Val = 3 : 1, 즉 75% : 25%)
    # 원래 6:2:2 의도에서 앞의 8(6+2)을 담당하므로 6 대 2 비율을 적용
    total_size = len(full_dataset)
    train_size = int(total_size * 0.75) 
    val_size = total_size - train_size

    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(
        full_dataset, [train_size, val_size], generator=generator
    )

    print(f"📊 데이터 분할 완료 | Train: {len(train_dataset)} (75%), Val: {len(val_dataset)} (25%)")

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    # 모델 준비
    model = SeqIDS(num_classes=5).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    weights = torch.tensor([1.0, 2.0, 2.0, 2.0, 2.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # Early Stopping 변수
    best_val_loss = float('inf')
    prev_val_loss = float('inf')
    overfit_count = 0
    overfit_limit = 2
    best_model_state = None
    
    print("\n🚀 학습 시작 (Early Stopping: Val Loss 증가 2회 감지 시 중단)")
    print("-" * 60)

    for epoch in range(1, epochs + 1):
        # [Training]
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits.permute(0, 2, 1).reshape(-1, 5),
                            y_batch.reshape(-1))

            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)

        # [Validation]
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits.permute(0, 2, 1).reshape(-1, 5), y_batch.reshape(-1))
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch [{epoch}/{epochs}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}", end="")

        # [Check Best Model]
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            print(" ✅ Best", end="")

        # [Check Overfitting]
        if epoch > 1 and avg_val_loss > prev_val_loss:
            overfit_count += 1
            print(f" ⚠️ Warning ({overfit_count}/{overfit_limit})", end="")
        else:
            # 연속이 아니어도 된다고 하셨지만, 보통 손실이 다시 줄어들면 카운트를 유지할지 초기화할지 결정해야 합니다.
            # "연속이 아니어도"라고 하셨으므로 초기화 코드는 넣지 않겠습니다.
            pass 
        
        print()

        if overfit_count >= overfit_limit:
            print(f"\n🛑 조기 종료! (과적합 징후 {overfit_limit}회 누적)")
            break
        
        prev_val_loss = avg_val_loss

    # Best Model 저장 및 복원
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        torch.save(model.state_dict(), model_save_path)
        print(f"💾 Best Model 저장 완료: {model_save_path}")
    
    return model

In [8]:
def test_seqids(model, eval_data_path, device):
    print(f"\n📂 평가 데이터 로드 중: {eval_data_path}")
    data_eval = np.load(eval_data_path)

    X_eval = torch.tensor(data_eval["X"], dtype=torch.float32)
    y_eval = torch.tensor(data_eval["y"], dtype=torch.long)

    eval_dataset = SeqWindowDataset(X_eval, y_eval)
    eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False)

    target_names = ['Normal', 'DoS', 'Fuzzy', 'Replay', 'Spoofing']
    evaluate_window_level(model, eval_loader, device, target_names)


In [10]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1) 경로 설정
    EVAL_DATA_PATH =  "C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/validation_dataset.npz"
    TRAIN_DATA_PATH  = "C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/dataset/test_dataset_all_CS.npz"

    MODEL_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/seqids_causal_final.pth"

    # 2) Train + Validation (Early Stopping)
    model = train_and_validate(
        train_data_path=TRAIN_DATA_PATH,
        model_save_path=MODEL_SAVE_PATH,
        epochs=20,
        lr=1e-3,
        device=device
    )

    # 3) 별도 평가 데이터셋으로 최종 평가
    print("\n=== [별도 평가 데이터셋 기준 윈도우 레벨 평가] ===")
    test_seqids(model, EVAL_DATA_PATH, device)

    print("💾 학습 및 평가 파이프라인이 완료되었습니다.")

🖥 실행 디바이스: cuda
📂 학습 데이터셋 로딩 중: C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/dataset/test_dataset_all_CS.npz
X shape: (117250, 9, 64) y shape: (117250, 64)
X dtype: float32 y dtype: int64
X has NaN: True | X has Inf: False
y has NaN: False | y has Inf: False
📊 데이터 분할 완료 | Train: 87937 (75%), Val: 29313 (25%)

🚀 학습 시작 (Early Stopping: Val Loss 증가 2회 감지 시 중단)
------------------------------------------------------------
Epoch [1/20] | Train Loss: 0.246784 | Val Loss: 0.127879 ✅ Best
Epoch [2/20] | Train Loss: 0.104912 | Val Loss: 0.101764 ✅ Best
Epoch [3/20] | Train Loss: 0.089630 | Val Loss: 0.083233 ✅ Best
Epoch [4/20] | Train Loss: 0.079307 | Val Loss: 0.079344 ✅ Best
Epoch [5/20] | Train Loss: 0.077629 | Val Loss: 0.077286 ✅ Best
Epoch [6/20] | Train Loss: 0.072479 | Val Loss: 0.073337 ✅ Best
Epoch [7/20] | Train Loss: 0.069811 | Val Loss: 0.072582 ✅ Best
Epoch [8/20] | Train Loss: 0.067721 | Val Loss: 0.069387 ✅ Best
Epoch [9/20] | Train Loss: 0.065884 | Val Loss: 0.066373 ✅ Bes